# 02 Pilot Local LLMs

            Objective: run a bounded pilot against one local OpenAI-compatible model and check JSON parse rate before the full experiment.

            By default this notebook does not call the model. Set `RUN_PILOT = True` after the local endpoint is running.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)

PROJECT_ROOT, CONFIG_PATH


## Configure Local Endpoint


In [ ]:
HOST = os.getenv("HOST", CONFIG["llm"]["host"])
            configured_models = [m.strip() for m in os.getenv("MODELS", ",".join(CONFIG["llm"]["models"])).split(",") if m.strip()]
            MODEL = os.getenv("MODEL", configured_models[0])
            RUN_PILOT = os.getenv("RUN_PILOT", "false").lower() in {"1", "true", "yes"}

            deterministic = CONFIG["llm"]["deterministic"]
            stochastic = CONFIG["llm"]["stochastic"]
            print({"HOST": HOST, "MODEL": MODEL, "RUN_PILOT": RUN_PILOT})


## Select Pilot Items


In [ ]:
benchmark = eu.read_csv_rows(PROJECT_ROOT / "data/processed/benchmark_items.csv")
            pilot_seed_count = int(CONFIG["project"]["pilot_seed_count"])
            pilot_seed_ids = sorted({row["seed_id"] for row in benchmark})[:pilot_seed_count]
            pilot_items = [row for row in benchmark if row["seed_id"] in pilot_seed_ids]
            planned_calls = len(pilot_items) * 2 * (1 + int(stochastic["samples"]))
            print(f"Pilot items: {len(pilot_items)} ({pilot_seed_count} seeds)")
            print(f"Planned calls for one model across both tasks: {planned_calls}")


## Run Pilot


In [ ]:
task1_template = eu.load_prompt(PROJECT_ROOT / "prompts/mandatory_entailment.txt")
task2_template = eu.load_prompt(PROJECT_ROOT / "prompts/modality_extraction.txt")

def prompt_for(task, item):
    if task == "task1":
        return eu.render_prompt(
            task1_template,
            source_statement=item["source_statement"],
            candidate_requirement=item["candidate_requirement"],
        )
    if task == "task2":
        return eu.render_prompt(task2_template, source_statement=item["source_statement"])
    raise ValueError(task)

def run_one(item, task, model, sample_kind, sample_index, temperature, top_p, output_path, run_id):
    prompt = prompt_for(task, item)
    completion = eu.chat_completion(
        host=HOST,
        model=model,
        prompt=prompt,
        temperature=temperature,
        top_p=top_p,
        max_tokens=int(CONFIG["llm"]["max_tokens"]),
        timeout_s=int(CONFIG["llm"]["timeout_s"]),
        api_key_env=CONFIG["llm"]["api_key_env"],
    )
    record = eu.build_raw_record(
        run_id=run_id,
        model=model,
        host=HOST,
        task=task,
        item=item,
        sample_index=sample_index,
        sample_kind=sample_kind,
        temperature=temperature,
        top_p=top_p,
        prompt_version=CONFIG["project"]["prompt_version"],
        prompt=prompt,
        completion=completion,
    )
    eu.append_jsonl(output_path, record)
    return record


In [ ]:
output_path = PROJECT_ROOT / "data/processed/model_outputs_raw_pilot.jsonl"
            run_id = eu.new_run_id("pilot")
            records = []

            if RUN_PILOT:
                for item in pilot_items:
                    for task in ["task1", "task2"]:
                        records.append(run_one(
                            item=item,
                            task=task,
                            model=MODEL,
                            sample_kind="deterministic",
                            sample_index=0,
                            temperature=float(deterministic["temperature"]),
                            top_p=float(deterministic["top_p"]),
                            output_path=output_path,
                            run_id=run_id,
                        ))
                        for sample_index in range(int(stochastic["samples"])):
                            records.append(run_one(
                                item=item,
                                task=task,
                                model=MODEL,
                                sample_kind="stochastic",
                                sample_index=sample_index,
                                temperature=float(stochastic["temperature"]),
                                top_p=float(stochastic["top_p"]),
                                output_path=output_path,
                                run_id=run_id,
                            ))
                print(f"Wrote {len(records)} pilot records to {output_path}")
            else:
                print("Pilot not run. Set RUN_PILOT=true in the environment or edit RUN_PILOT to True.")


## Pilot Gate


In [ ]:
pilot_rows = [row for row in eu.read_jsonl(output_path) if row.get("run_id") == run_id] if RUN_PILOT else []
            if pilot_rows:
                ok = sum(1 for row in pilot_rows if row["parse_status"] == "ok")
                parse_rate = ok / len(pilot_rows)
                avg_latency = sum(float(row["latency_s"] or 0) for row in pilot_rows) / len(pilot_rows)
                print(f"Parse success: {ok}/{len(pilot_rows)} = {parse_rate:.3f}")
                print(f"Average latency: {avg_latency:.2f}s")
                if parse_rate < 0.95:
                    print("Gate failed: inspect invalid outputs before the full run.")
                else:
                    print("Gate passed: parse success is >= 95%.")
